In [ ]:
# Etapa 1 — imports mínimos e parâmetros
import os
import sys

import pandas as pd

sys.path.append(os.path.abspath('..'))

from analise_liquidez.dados import TITULOS_PUBLICOS_SCHEMA
from analise_liquidez.graficos import plot_titulo_plotly
from analise_liquidez.notebook_pipeline import (
    aplicar_filtro_outlier_quantidade,
    calcular_top_negociados,
    carregar_dados_ativo_com_cache,
    gerar_grafico_top_quantidade_media,
    preparar_historico_expandido,
    preparar_recorte_recente,
)
from analise_liquidez.regras import calcular_liquidez_carteira, calcular_medias_e_volumes

URL = 'https://github.com/PulseDataLabs/PulseFlat/raw/refs/heads/main/data/bacen_negociacao_tpf_extragrupo.csv.gz'
LOCAL_PATHS = [
    '../../PulseFlat/data/bacen_negociacao_tpf_extragrupo.csv.gz',
    '../PulseFlat/data/bacen_negociacao_tpf_extragrupo.csv.gz',
]

DATA_REFERENCIA = pd.to_datetime('2026-07-31')
JANELA_MESES = 3
TOP_N = 5
TOP_N_BARRA = 10
OUTLIER_QUANTILE = 0.999

DIR_CACHE = os.path.join(os.path.abspath('..'), 'data')
CACHE_DADOS = os.path.join(DIR_CACHE, 'titulos_publicos_ajustado_cache.csv.gz')
CACHE_EXPANDIDO = os.path.join(DIR_CACHE, 'titulos_publicos_expandido_cache.csv.gz')
COLUNAS_DESCARTAR = ['arquivo_origem', 'data_captura', 'conjunto', 'registro_hash']


In [ ]:
# Etapa 2 — carregamento de dados com cache

df_dados = carregar_dados_ativo_com_cache(
    schema=TITULOS_PUBLICOS_SCHEMA,
    cache_path=CACHE_DADOS,
    url=URL,
    local_paths=LOCAL_PATHS,
    drop_cols=COLUNAS_DESCARTAR,
)

print(
    f"Período bruto: {df_dados['data_mov'].min().date()} a "
    f"{df_dados['data_mov'].max().date()}"
)
print(f"ISINs analisados: {df_dados['codigo_isin'].nunique():,}")
print(f"Linhas: {len(df_dados):,}")
display(df_dados.head())


In [ ]:
# Etapa 3 — preparação (dias úteis, recorte temporal e outlier)

df_expanded = preparar_historico_expandido(
    df=df_dados,
    schema=TITULOS_PUBLICOS_SCHEMA,
    cache_path=CACHE_EXPANDIDO,
)

df_analise_recent = preparar_recorte_recente(
    df=df_expanded,
    schema=TITULOS_PUBLICOS_SCHEMA,
    data_referencia=DATA_REFERENCIA,
    meses=JANELA_MESES,
)

df_analise_recent = aplicar_filtro_outlier_quantidade(
    df=df_analise_recent,
    schema=TITULOS_PUBLICOS_SCHEMA,
    quantile=OUTLIER_QUANTILE,
)

display(df_analise_recent.head())


In [ ]:
# Etapa 4 — métricas e tabelas de apoio

top_isins, top_info = calcular_top_negociados(
    df=df_analise_recent,
    schema=TITULOS_PUBLICOS_SCHEMA,
    top_n=TOP_N,
)

print('Top títulos públicos por número de negócios no período recente:')
display(top_info)

df_analise = calcular_medias_e_volumes(df_expanded, TITULOS_PUBLICOS_SCHEMA)
display(
    df_analise.sort_values(
        by=['data_mov', 'quantidade_media_21_dias'],
        ascending=[False, False],
    ).head()
)


In [ ]:
# Etapa 5 — visualização temporal do ativo mais negociado

if top_isins:
    isin_mais_negociado = top_isins[0]
    codigo_ativo = top_info.loc[top_info['codigo_isin'] == isin_mais_negociado, 'codigo'].iloc[0]

    fig_plotly = plot_titulo_plotly(
        df_data=df_analise_recent,
        isin_to_plot=isin_mais_negociado,
        codigo_ativo_to_plot=codigo_ativo,
        schema=TITULOS_PUBLICOS_SCHEMA,
        y_column='num_de_oper',
        title_suffix='(Últimos 3 Meses)',
        window_size=7,
        window_size_secondary=30,
    )
    fig_plotly.show()
else:
    print('Sem ativos no recorte para plotagem.')


In [ ]:
# Etapa 6 — ranking final por liquidez média (21 dias)

top_10_info_media, fig_bar = gerar_grafico_top_quantidade_media(
    df_analise=df_analise,
    schema=TITULOS_PUBLICOS_SCHEMA,
    top_n=TOP_N_BARRA,
    title='Top 10 Títulos Públicos por Quantidade Média de 21 Dias',
)

display(top_10_info_media)
fig_bar.show()


In [ ]:
# Etapa 7 — análise de liquidez de carteira

parametros_calculo = {
    'data_referencia': DATA_REFERENCIA,
    'prazo_de_cotizacao': 1,
    'carteira_de_titulos_publicos': [
        ['BRSTNCNTB4U6', 'NTB4U6', 4000],
        ['BRSTNCNTB4X0', 'NTB4X0', 108353],
        ['BRSTNCNTB633', 'NTB633', 27048],
        ['BRSTNCNTB3B8', 'NTB3B8', 27048],
        ['BRSTNCNTB682', 'NTB682', 27048],
    ],
}

df_analise_liquidez_carteira = calcular_liquidez_carteira(
    parametros=parametros_calculo,
    df_dados_liquidez=df_analise,
    schema=TITULOS_PUBLICOS_SCHEMA,
)

print('\n--- Resultado da Análise de Liquidez da Carteira ---')
pd.options.display.float_format = '{:,.2f}'.format
display(df_analise_liquidez_carteira.head())
